# Начальные и граничные условия

Рассмотрим ряд классов для определения начальных и граничных условия **для задачи переноса изотопов водорода** — от простых условий до поверхностной диссоциации и рекомбинации.

## Что выбирать

- Известна концентрация на поверхности → `FixedConcentrationBC`.
- Известен внешний поток → `ParticleFluxBC`.
- Поверхность находится в равновесии с газом → `HenrysBC` или `SievertsBC`.
- Нужно учитывать диссоциацию и рекомбинацию → `SurfaceReactionBC`.

В один расчёт добавляется только физически согласованный набор граничных условий. Если на поверхности не задано граничное условие, FESTIM использует естественное условие нулевого потока:

$$
\mathbf{J}\cdot\mathbf{n}=0.
$$

## Общая одномерная область

Создадим простую область, один материал, две поверхности и один мобильный вид частиц.

In [ ]:
import festim as F
import numpy as np

L = 1e-3

model = F.HydrogenTransportProblem()

model.mesh = F.Mesh1D(
    vertices=np.linspace(0.0, L, 101),
)

material = F.Material(
    D_0=1.11e-6,
    E_D=0.4,
)

volume = F.VolumeSubdomain1D(
    id=1,
    borders=[0.0, L],
    material=material,
)

left = F.SurfaceSubdomain1D(id=1, x=0.0)
right = F.SurfaceSubdomain1D(id=2, x=L)

model.subdomains = [volume, left, right]

H = F.Species("H")
model.species = [H]

## Начальная концентрация

Начальные условия используются только в нестационарных задачах. Если они не заданы, начальная концентрация равна нулю.

Значение может быть константой или функцией координат и температуры.

In [ ]:
# Постоянная начальная концентрация
initial_constant = F.InitialConcentration(
    value=1e18,
    species=H,
    volume=volume,
)

# Пример неоднородного начального профиля
initial_profile = F.InitialConcentration(
    value=lambda x: 1e20 * (1.0 - x[0] / L),
    species=H,
    volume=volume,
)

# Для расчёта выбирается один из вариантов
model.initial_conditions = [initial_constant]

## Фиксированная концентрация на границе

`FixedConcentrationBC` задаёт значение концентрации на поверхности:

$$
c = c_{\mathrm{BC}}.
$$

Значение может зависеть от координат `x`, времени `t` и температуры `T`.

In [ ]:
fixed_left = F.FixedConcentrationBC(
    subdomain=left,
    value=1e20,
    species=H,
)

time_dependent_left = F.FixedConcentrationBC(
    subdomain=left,
    value=lambda t: 1e20 + 1e18 * t,
    species=H,
)

## Заданный поток частиц

`ParticleFluxBC` задаёт нормальную составляющую потока:

$$
\mathbf{J}\cdot\mathbf{n}=g.
$$

In [ ]:
# Постоянный поток внутрь материала
implantation_flux = F.ParticleFluxBC(
    subdomain=right,
    value=-1e20,
    species=H,
)

# Поток, зависящий от концентрации на поверхности
linear_outgassing = F.ParticleFluxBC(
    subdomain=right,
    value=lambda c: 1e-3 * c,
    species=H,
    species_dependent_value={"c": H},
)

model.boundary_conditions = [
    fixed_left,
    linear_outgassing,
]

## Равновесие с газовой фазой

`HenrysBC` и `SievertsBC` — удобные варианты фиксированной концентрации, рассчитанной из давления газа.

Закон Генри:

$$
c = H(T)P.
$$

Закон Сивертса:

$$
c = S(T)\sqrt{P}.
$$

In [ ]:
henrys_left = F.HenrysBC(
    subdomain=left,
    H_0=1.5,
    E_H=0.2,
    pressure=1e5,
    species=H,
)

sieverts_left = F.SievertsBC(
    subdomain=left,
    S_0=2.0,
    E_S=0.1,
    pressure=1e5,
    species=H,
)

## Диссоциация и рекомбинация на поверхности

Стандартную обратимую поверхностную реакцию

$$
\mathrm{H + H \rightleftharpoons H_2}
$$

можно задать с помощью `SurfaceReactionBC`.

FESTIM рассчитывает результирующую скорость реакции и автоматически добавляет соответствующий поток атомарного водорода на границе. Отдельная поверхностная концентрация при этом не вводится.

In [ ]:
surface_reaction = F.SurfaceReactionBC(
    reactant=[H, H],
    gas_pressure=1e5,
    k_r0=1e-25,
    E_kr=0.8,
    k_d0=1e5,
    E_kd=1.0,
    subdomain=right,
)

# Пример набора условий:
# слева задана концентрация,
# справа происходят диссоциация и рекомбинация
model.boundary_conditions = [
    fixed_left,
    surface_reaction,
]

## Простой стационарный расчёт

Проверим два фиксированных граничных условия:

$$
c(0)=10^{20}\ \mathrm{м^{-3}}, \qquad c(L)=0.
$$

Для однородного материала и постоянной температуры стационарный профиль концентрации должен быть линейным.

Начальное условие в стационарной задаче не используется, поэтому перед запуском очистим `model.initial_conditions`.

In [ ]:
model.initial_conditions = []

model.boundary_conditions = [
    F.FixedConcentrationBC(
        subdomain=left,
        value=1e20,
        species=H,
    ),
    F.FixedConcentrationBC(
        subdomain=right,
        value=0.0,
        species=H,
    ),
]

model.temperature = 400.0

model.settings = F.Settings(
    atol=1e-10,
    rtol=1e-8,
    transient=False,
)

model.initialise()
model.run()


## Визуализация

In [ ]:
import matplotlib.pyplot as plt

x = H.post_processing_solution.function_space.tabulate_dof_coordinates()[:, 0]
c = H.post_processing_solution.x.array
order = np.argsort(x)

x_sorted = x[order]
c_sorted = c[order]

c_exact = 1e20 * (1.0 - x_sorted / L)

plt.figure(figsize=(7, 4))
plt.plot(x_sorted * 1e3, c_sorted, label="FESTIM")
plt.plot(
    x_sorted * 1e3,
    c_exact,
    "--",
    label="Линейный профиль",
)
plt.xlabel("Координата, мм")
plt.ylabel("Концентрация H, м$^{-3}$")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
